In [1]:
import numpy as np
import ee
import geemap

ee.Authenticate()
ee.Initialize()

In [ ]:
map = geemap.Map()
map.add_basemap("HYBRID")
center = ee.Geometry.Point([46.439209, 30.949347])
map.centerObject(center, 4)

map

Map(center=[30.949347000000003, 46.439209], controls=(WidgetControl(options=['position', 'transparent_bg'], po…

In [ ]:
roi = map.draw_last_feature.geometry()
print("ROI geometry:", roi)

In [19]:
# Creating a timelist
time_start = ee.Date('2005')
time_end = ee.Date('2010')
time_dif = time_end.difference(time_start, 'year').round()
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(
    lambda x: time_start.advance(x, 'year')
)

# Function for obtaining the date for the spicfic time
def annual(date, col):
  start_date = ee.Date(date)
  end_date = start_date.advance(1, 'year')
  img_sum = col.filterDate(start_date, end_date).sum()
  return img_sum.set('system:time_start', start_date.millis())

In [23]:
vis_params4 =  {'bands': ['NDVI'], 'min': -7970.3, 'max': 87319.55, 'palette': ['#f7fcf5', '#c7e9c0', '#74c476', '#238b45', '#00441b']}

In [24]:
# Script in the cell is Demo
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .filterDate(time_start, time_end)
    .select("NDVI")
    .map(lambda x: x.multiply(0.0001).copyProperties(x, ['system:time_start'])
    )
)

ndvi_annual = ee.ImageCollection(time_list.map(lambda x: annual(date=x, col=ndvi)))

In [25]:
map.addLayer(ndvi.select(['NDVI']).mean().clip(roi), vis_params4, "NDVI")

<IPython.core.display.Javascript object>

In [ ]:
# Add one NDVI layer for each year (toggle in the layer manager)
annual_list = ndvi_annual.toList(ndvi_annual.size())
annual_count = ndvi_annual.size().getInfo()

for i in range(5):
    annual_img = ee.Image(annual_list.get(i))
    year_label = ee.Date(annual_img.get('system:time_start')).format('YYYY').getInfo()
    map.addLayer(
        annual_img.clip(roi),
        {'bands': ['NDVI'], 'min': -7970.3, 'max': 87319.55, 'palette': ['#f7fcf5', '#c7e9c0', '#74c476', '#238b45', '#00441b']},
        f'NDVI {year_label}',
        False
    )

map

Map(bottom=27212.0, center=[30.600093873550072, 47.30712890625001], controls=(WidgetControl(options=['position…